In [1]:
# Cell 1: Install packages and Cloudflare Tunnel
!pip install -q streamlit plotly pyngrok
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Environment & Tunneling Ready!")

✅ Environment & Tunneling Ready!


In [2]:
# Cell 2: Auto-detect files from your previous runs
from pathlib import Path

splat_ply = next(Path("/kaggle").rglob("ceramic_splatfacto*.ply"), None)
nerf_ply = next(Path("/kaggle").rglob("point_cloud.ply"), None)
results_csv = next(Path("/kaggle").rglob("results.csv"), None)

print("🔍 Detected Files in Kaggle:")
print(" - Splatfacto 3D Model (.ply) :", splat_ply or "Not found (Demo mode will be used)")
print(" - Nerfacto Point Cloud (.ply):", nerf_ply or "Not found (Demo mode will be used)")
print(" - Metrics results.csv        :", results_csv or "Built-in verified metrics will be used")

🔍 Detected Files in Kaggle:
 - Splatfacto 3D Model (.ply) : /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/ceramic_splatfacto.ply
 - Nerfacto Point Cloud (.ply): /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/point_cloud.ply
 - Metrics results.csv        : Built-in verified metrics will be used


In [3]:
%%writefile app.py
import os
import json
import base64
from pathlib import Path
import numpy as np
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
import plotly.graph_objects as go

st.set_page_config(
    page_title="3D Cultural Artifact Preservation",
    page_icon="🏺",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Styling
st.markdown("""
<style>
    .stApp { background-color: #0b1320; color: #e6edf3; }
    .metric-card {
        background: linear-gradient(135deg, #132238 0%, #1a2f4c 100%);
        border: 1px solid #233e66; border-radius: 10px; padding: 14px 18px; margin-bottom: 10px;
    }
    .metric-title { color: #8da4c4; font-size: 12px; font-weight: 600; text-transform: uppercase; }
    .metric-value { color: #ffffff; font-size: 24px; font-weight: 700; margin: 3px 0; }
    .metric-delta { color: #3fb950; font-size: 12px; font-weight: 600; }
</style>
""", unsafe_allow_html=True)

# 1. Check auto-discovered files from Kaggle
splat_auto = next(Path("/kaggle").rglob("ceramic_splatfacto*.ply"), None)
nerf_auto = next(Path("/kaggle").rglob("point_cloud*.ply"), None)

# 2. Sidebar with DUAL file uploaders + MULTI-FILE drag-and-drop
with st.sidebar:
    st.image("https://img.icons8.com/fluency/96/vase.png", width=54)
    st.title("Preservation Hub")
    st.caption("Upload Both Models (Splatfacto + Nerfacto)")
    st.markdown("---")
    
    st.subheader("📁 Upload Your 3D Files")
    # Multiple files upload (drag both at once!)
    multi_uploaded = st.file_uploader(
        "Drag & Drop BOTH .ply files here:",
        type=["ply"],
        accept_multiple_files=True,
        help="You can select both ceramic_splatfacto.ply and point_cloud.ply together!"
    )
    
    # Or dedicated individual slots
    with st.expander("Or upload individually:"):
        upload_splat = st.file_uploader("1. Splatfacto (3DGS .ply):", type=["ply"], key="u_splat")
        upload_nerf = st.file_uploader("2. Nerfacto (NeRF .ply):", type=["ply"], key="u_nerf")

# 3. Resolve which file is which
splat_bytes = None
nerf_bytes = None

# Check multi-upload first
if multi_uploaded:
    for f in multi_uploaded:
        fname = f.name.lower()
        if "splat" in fname:
            splat_bytes = f.getvalue()
            st.sidebar.success(f"Loaded 3DGS: {f.name}")
        elif "nerf" in fname or "point" in fname:
            nerf_bytes = f.getvalue()
            st.sidebar.success(f"Loaded NeRF: {f.name}")
        else:
            # Default first to splat, second to nerf
            if splat_bytes is None: splat_bytes = f.getvalue()
            elif nerf_bytes is None: nerf_bytes = f.getvalue()

# Check individual uploaders
if upload_splat:
    splat_bytes = upload_splat.getvalue()
if upload_nerf:
    nerf_bytes = upload_nerf.getvalue()

# Fallback to auto-detected Kaggle files
if splat_bytes is None and splat_auto:
    splat_bytes = splat_auto.read_bytes()
if nerf_bytes is None and nerf_auto:
    nerf_bytes = nerf_auto.read_bytes()

# Encode to base64 for WebGL renderer
splat_b64 = base64.b64encode(splat_bytes).decode("ascii") if splat_bytes else ""
nerf_b64 = base64.b64encode(nerf_bytes).decode("ascii") if nerf_bytes else ""

# Status summary in sidebar
st.sidebar.markdown("---")
st.sidebar.markdown(f"**Splatfacto Model:** {'`Loaded ✅`' if splat_bytes else '`Demo Vase`'}")
st.sidebar.markdown(f"**Nerfacto Model:** {'`Loaded ✅`' if nerf_bytes else '`Demo Vase`'}")

# Header
st.title("🏺 Digital Preservation of Cultural Artifacts")
st.markdown("Comparative 3D Pipeline: **NeRF (Nerfacto)** vs **3D Gaussian Splatting (Splatfacto)**")

# Top Metrics
c1, c2, c3, c4 = st.columns(4)
with c1:
    st.markdown('<div class="metric-card"><div class="metric-title">PSNR Quality</div><div class="metric-value">20.37 dB</div><div class="metric-delta">▲ +8.40 dB vs NeRF</div></div>', unsafe_allow_html=True)
with c2:
    st.markdown('<div class="metric-card"><div class="metric-title">SSIM Score</div><div class="metric-value">0.8270</div><div class="metric-delta">▲ +26.4% fidelity</div></div>', unsafe_allow_html=True)
with c3:
    st.markdown('<div class="metric-card"><div class="metric-title">LPIPS Error</div><div class="metric-value">0.2910</div><div class="metric-delta">▼ -45.7% error</div></div>', unsafe_allow_html=True)
with c4:
    st.markdown('<div class="metric-card"><div class="metric-title">Rendering Speed</div><div class="metric-value">18.20 FPS</div><div class="metric-delta">⚡ 165x speedup</div></div>', unsafe_allow_html=True)

# Tabs
t_viewer, t_bench, t_renders, t_cameras = st.tabs(["🌐 3D Web Viewer (Both Models)", "📊 Benchmark Comparison", "🖼️ Held-Out Views", "📷 Camera Orbits"])

# TAB 1: 3D DUAL VIEWER
with t_viewer:
    st.subheader("Dual Model 3D WebGL Viewer (Switch between Splatfacto & Nerfacto)")
    viewer_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    <style>
      body {{ margin:0; background:#080e18; color:white; font-family:sans-serif; overflow:hidden; }}
      .bar {{ display:flex; gap:8px; padding:10px; background:#101c2d; border-bottom:1px solid #1f3352; align-items:center; }}
      button {{ padding:6px 14px; background:#18283f; color:#e2ecf8; border:1px solid #2b456b; border-radius:5px; cursor:pointer; font-size:12px; font-weight:500; }}
      button.active {{ background:#1f6feb; border-color:#58a6ff; color:white; box-shadow:0 0 8px rgba(31,111,235,0.4); }}
      #viewer {{ width:100%; height:540px; position:relative; }}
      .hud {{ position:absolute; top:10px; right:12px; background:rgba(13,22,36,0.85); padding:8px 12px; border-radius:6px; font-size:11px; color:#79c0ff; border:1px solid #388bfd40; pointer-events:none; }}
    </style>
    <script type="importmap">
    {{
      "imports": {{
        "three": "https://cdn.jsdelivr.net/npm/three@0.180.0/build/three.module.js",
        "three/addons/": "https://cdn.jsdelivr.net/npm/three@0.180.0/examples/jsm/",
        "@sparkjsdev/spark": "https://sparkjs.dev/releases/spark/2.4.0/spark.module.js"
      }}
    }}
    </script>
    </head>
    <body>
    <div class="bar">
      <button id="splatBtn" class="active">✨ Splatfacto (Gaussian Splats)</button>
      <button id="nerfBtn">🔵 Nerfacto (Point Cloud)</button>
      <button id="demoBtn">🏺 Procedural Demo</button>
      <div style="flex:1"></div>
      <button id="frontBtn">Front</button><button id="topBtn">Top</button><button id="sideBtn">Side</button><button id="resetBtn">Reset</button>
      <button id="rotBtn">🔄 Orbit: Off</button>
    </div>
    <div id="viewer"><div class="hud" id="hud">Loading Models...</div></div>

    <script type="module">
      import * as THREE from "three";
      import {{ OrbitControls }} from "three/addons/controls/OrbitControls.js";
      import {{ PLYLoader }} from "three/addons/loaders/PLYLoader.js";
      import {{ SparkRenderer, SplatMesh }} from "@sparkjsdev/spark";

      const host = document.getElementById("viewer");
      const hud = document.getElementById("hud");
      const scene = new THREE.Scene();
      scene.background = new THREE.Color(0x080e18);
      const camera = new THREE.PerspectiveCamera(48, host.clientWidth / host.clientHeight, 0.001, 1000);
      const renderer = new THREE.WebGLRenderer({{ antialias:true }});
      renderer.setSize(host.clientWidth, host.clientHeight);
      host.appendChild(renderer.domElement);

      let spark = null;
      try {{ spark = new SparkRenderer({{ renderer }}); scene.add(spark); }} catch(e) {{}}
      const controls = new OrbitControls(camera, renderer.domElement);
      controls.enableDamping = true;
      scene.add(new THREE.GridHelper(3, 30, 0x1f6feb, 0x15273f));
      scene.add(new THREE.AmbientLight(0xffffff, 1.2));

      let splatObj = null, nerfObj = null, demoObj = null, autoRot = false;
      const splatData = "{splat_b64}";
      const nerfData = "{nerf_b64}";

      function createDemo() {{
        const n = 12000, geo = new THREE.BufferGeometry(), pos = new Float32Array(n*3), col = new Float32Array(n*3);
        for(let i=0; i<n; i++) {{
          const t = Math.random(), th = Math.random() * Math.PI * 2;
          const r = 0.22 + 0.35 * Math.sin(Math.PI * Math.pow(t, 0.8)) - 0.12 * Math.cos(3*Math.PI*t);
          pos[i*3] = r * Math.cos(th); pos[i*3+1] = (t-0.5)*1.4; pos[i*3+2] = r * Math.sin(th);
          col[i*3] = 0.2 + 0.5*(1-t); col[i*3+1] = 0.4 + 0.5*(1-t); col[i*3+2] = 0.7 + 0.3*t;
        }}
        geo.setAttribute("position", new THREE.BufferAttribute(pos, 3));
        geo.setAttribute("color", new THREE.BufferAttribute(col, 3));
        demoObj = new THREE.Points(geo, new THREE.PointsMaterial({{ size:0.012, vertexColors:true }}));
        scene.add(demoObj);
      }}

      async function init() {{
        createDemo();
        // Load Splatfacto
        if (splatData) {{
          try {{
            splatObj = new SplatMesh({{ url: "data:application/octet-stream;base64," + splatData, lod: true }});
            scene.add(splatObj);
            await splatObj.initialized;
            demoObj.visible = false;
            splatObj.visible = true;
            hud.textContent = "Mode: Splatfacto (3D Gaussian Splats Loaded)";
          }} catch(e) {{ console.log(e); }}
        }}
        // Load Nerfacto
        if (nerfData) {{
          try {{
            const geo = await new PLYLoader().loadAsync("data:application/octet-stream;base64," + nerfData);
            nerfObj = new THREE.Points(geo, new THREE.PointsMaterial({{ size:0.005, vertexColors:true }}));
            nerfObj.visible = false;
            scene.add(nerfObj);
          }} catch(e) {{ console.log(e); }}
        }}
        camera.position.set(0, 0.2, 2.5);
      }}

      function setMode(m) {{
        document.getElementById("splatBtn").classList.toggle("active", m==="splat");
        document.getElementById("nerfBtn").classList.toggle("active", m==="nerf");
        document.getElementById("demoBtn").classList.toggle("active", m==="demo");
        if(splatObj) splatObj.visible = (m==="splat");
        if(nerfObj) nerfObj.visible = (m==="nerf");
        if(demoObj) demoObj.visible = (m==="demo" || (!splatObj && m==="splat") || (!nerfObj && m==="nerf"));
        hud.textContent = `Mode: ${{m === "splat" ? "Splatfacto (3DGS)" : m === "nerf" ? "Nerfacto (NeRF Points)" : "Procedural Demo"}}`;
      }}

      document.getElementById("splatBtn").onclick = () => setMode("splat");
      document.getElementById("nerfBtn").onclick = () => setMode("nerf");
      document.getElementById("demoBtn").onclick = () => setMode("demo");
      document.getElementById("frontBtn").onclick = () => {{ camera.position.set(0, 0.2, 2.5); controls.target.set(0,0,0); }};
      document.getElementById("topBtn").onclick = () => {{ camera.position.set(0.001, 2.8, 0); controls.target.set(0,0,0); }};
      document.getElementById("sideBtn").onclick = () => {{ camera.position.set(2.8, 0.1, 0); controls.target.set(0,0,0); }};
      document.getElementById("resetBtn").onclick = () => {{ camera.position.set(0, 0.2, 2.5); controls.reset(); }};
      document.getElementById("rotBtn").onclick = (e) => {{
        autoRot = !autoRot; controls.autoRotate = autoRot;
        e.currentTarget.textContent = autoRot ? "🔄 Orbit: On" : "🔄 Orbit: Off";
        e.currentTarget.classList.toggle("active", autoRot);
      }};

      window.onresize = () => {{
        camera.aspect = host.clientWidth / host.clientHeight;
        camera.updateProjectionMatrix();
        renderer.setSize(host.clientWidth, host.clientHeight);
      }};

      let frames = 0, last = performance.now();
      renderer.setAnimationLoop(() => {{
        controls.update();
        renderer.render(scene, camera);
        frames++;
        const now = performance.now();
        if(now - last >= 1000) {{
          const fps = Math.round((frames * 1000)/(now - last));
          hud.textContent = hud.textContent.split("|")[0] + `| FPS: ${{fps}}`;
          frames = 0; last = now;
        }}
      }});
      init();
    </script>
    </body>
    </html>
    """
    components.html(viewer_html, height=580, scrolling=False)
    st.caption("Controls: Left-drag to Orbit · Right-drag to Pan · Scroll wheel to Zoom.")

# TAB 2: BENCHMARKS
with t_bench:
    st.subheader("Quantitative Evaluation Results (18 Held-Out Views)")
    df = pd.DataFrame([
        {"Metric": "PSNR (dB)", "Nerfacto (NeRF)": 11.9713, "Splatfacto (3DGS v1)": 20.1740, "Splatfacto v2 (OBJ Init)": 20.3700, "Winner / Gain": "+8.40 dB (+70.2%)"},
        {"Metric": "SSIM", "Nerfacto (NeRF)": 0.6542, "Splatfacto (3DGS v1)": 0.8172, "Splatfacto v2 (OBJ Init)": 0.8270, "Winner / Gain": "+0.1728 (+26.4%)"},
        {"Metric": "LPIPS", "Nerfacto (NeRF)": 0.5364, "Splatfacto (3DGS v1)": 0.2979, "Splatfacto v2 (OBJ Init)": 0.2910, "Winner / Gain": "-45.7% error reduction"},
        {"Metric": "FPS (Evaluation)", "Nerfacto (NeRF)": 0.1050, "Splatfacto (3DGS v1)": 17.3500, "Splatfacto v2 (OBJ Init)": 18.2000, "Winner / Gain": "165.2x faster rendering"},
        {"Metric": "Exported Web PLY", "Nerfacto (NeRF)": 28.40, "Splatfacto (3DGS v1)": 14.80, "Splatfacto v2 (OBJ Init)": 1.82, "Winner / Gain": "15.6x smaller (Cleaned)"}
    ])
    st.dataframe(df, hide_index=True, use_container_width=True)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=["Nerfacto", "Splatfacto v1", "Splatfacto v2"], y=[11.97, 20.17, 20.37], name="PSNR (dB)", marker_color=["#388bfd", "#f0883e", "#3fb950"]))
    fig.update_layout(title="PSNR (dB) Quality Comparison", template="plotly_dark", paper_bgcolor="#0c1422", plot_bgcolor="#0c1422")
    st.plotly_chart(fig, use_container_width=True)

# TAB 3: HELD-OUT RENDERS
with t_renders:
    st.subheader("Novel View Synthesis on Held-Out Test Views")
    st.caption("Side-by-side reconstruction across the 18 evaluation cameras.")
    selected_idx = st.slider("Select View Index (0 - 17):", 0, 17, 0)
    nerf_img = next(Path("/kaggle").rglob(f"eval_renders/*{selected_idx:04d}*.png"), None)
    splat_img = next(Path("/kaggle").rglob(f"splatfacto_eval_renders/*{selected_idx:04d}*.png"), None)

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("**Nerfacto (NeRF)**")
        if nerf_img: st.image(str(nerf_img))
        else: st.info(f"eval_img_{selected_idx:04d}.png will appear once attached.")
    with col2:
        st.markdown("**Splatfacto (3DGS)**")
        if splat_img: st.image(str(splat_img))
        else: st.info(f"splatfacto_img_{selected_idx:04d}.png will appear once attached.")

# TAB 4: CAMERAS
with t_cameras:
    st.subheader("Camera Rigs (CAM1 vs CAM2)")
    st.markdown("""
    - **Total Views:** 180 (90 CAM1 at +0.45 elevation + 90 CAM2 at -0.25 elevation)
    - **Focal Length:** $f_x = f_y = 711.1235$, $c_x = c_y = 256.0$
    - **Prior Geometry:** `ceramic_vases_1.obj` (100,000 surface vertices)
    """)

Writing app.py


In [10]:
# Cell 4: Launch Streamlit & Start Free Cloudflare Tunnel
import subprocess, time, re

# Kill old instances if any
!pkill -f streamlit || true

# Start Streamlit in background
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true", shell=True)
time.sleep(3)

# Start Cloudflare Tunnel to expose the app publicly
proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stderr=subprocess.PIPE, text=True)

# Read the generated URL
url = None
for _ in range(25):
    line = proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
    time.sleep(0.5)

print("\n" + "="*60)
if url:
    print("🎉 YOUR STREAMLIT APP IS RUNNING LIVE!")
    print(f"👉 CLICK THIS URL TO OPEN: {url}")
else:
    print("Could not auto-detect tunnel URL. Check if Internet is turned ON.")
print("="*60 + "\n")

  Stopping...




2026-11-15 09:10:22.791 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://35.185.152.118:8501


🎉 YOUR STREAMLIT APP IS RUNNING LIVE!
👉 CLICK THIS URL TO OPEN: https://creating-option-sufficiently-bend.trycloudflare.com



2026-11-15 09:10:34.823 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-11-15 09:10:34.825 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-11-15 09:10:35.303 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-11-15 09:10:41.144 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-11-15 09:10:41.146 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width=